# 🦙 1.6 Open Source LLMs with LangChain

## Learning Objectives
In this notebook, you will learn how to run open-source LLMs through LangChain using three different backends:
1. **Local inference** - `HuggingFacePipeline` + `ChatHuggingFace` to run a model on your own machine (CPU/GPU/MPS)
2. **Hosted inference** - `HuggingFaceEndpoint` + `ChatHuggingFace` to call a Hugging Face-hosted inference endpoint
3. **Groq-hosted inference** - `ChatGroq` for fast, free-tier inference of open-source models
4. **A common LCEL pattern** - the same `prompt | chat_model` chain works unchanged across all three backends

## Prerequisites
- `langchain-huggingface` and `langchain-groq` installed
- `GROQ_API_KEY` in your project `.env` for the Groq section
- A local GPU/MPS device recommended (but not required) for the local-inference section


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

## Use LLMs locally with LangChain and Hugging Face

`HuggingFacePipeline.from_model_id(...)` downloads the model and runs it in-process using
Hugging Face `transformers`. Wrapping it in `ChatHuggingFace` gives it LangChain's chat
model interface (message-in, `AIMessage`-out), so it can be used in an LCEL chain exactly
like a hosted chat model.

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

model_id = "moonshotai/kimi-k2-instruct-0905"

llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=1000,
        do_sample=False,
        temperature=0,
        return_full_text=False,
    ),
    device_map="auto"  # Automatically selects best device (MPS on Mac, CUDA on GPU, CPU as fallback)
)
llm.pipeline.tokenizer.pad_token_id = llm.pipeline.tokenizer.eos_token_id
chat_llama = ChatHuggingFace(llm=llm)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

## Use LLMs with LangChain and Hugging Face Inference APIs

Instead of downloading weights, `HuggingFaceEndpoint` calls a model hosted by Hugging Face's
Inference API/Endpoints over the network. Same `ChatHuggingFace` wrapper as the local case, so
the rest of the code — building the prompt, chaining it, invoking it — is identical.

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

model_id = "moonshotai/kimi-k2-instruct-0905"

llm_api = HuggingFaceEndpoint(
    repo_id=model_id,
    task="text-generation",
    max_new_tokens=1000,
    do_sample=False,
    temperature=0,
)

chat_llama = ChatHuggingFace(llm=llm_api)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

## Load Groq API Credentials

Groq's free tier serves open-source models (Llama, Mixtral, etc.) at very low latency via its
own LPU hardware. As with the other notebooks in this folder, the API key is loaded from
`.env` rather than hardcoded.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

## Use LLMs with LangChain and Groq API

`ChatGroq` is a native LangChain chat model — no `ChatHuggingFace` wrapper needed, since Groq
already exposes an OpenAI-compatible chat completion API that LangChain integrates with directly.

In [ ]:
from langchain_groq import ChatGroq

chat_llama = ChatGroq(
    model="llama-3.2-3b-preview",
    temperature=0,
    max_tokens=1000,
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

PROMPT = "Explain {topic} in 2 bullets"
prompt = ChatPromptTemplate.from_template(PROMPT)

chain = (
         prompt
           |
         chat_llama
)

response = chain.invoke({"topic": "AI"})
print(response.content)

---
## 📝 Summary

In this notebook, we learned:

### 1. Local Inference
- **Key point**: `HuggingFacePipeline.from_model_id(...)` runs an open-source model in-process (CPU/GPU/MPS)
- **Key point**: `ChatHuggingFace` wraps any HF LLM object to give it LangChain's chat model interface

### 2. Hosted Hugging Face Inference
- **Key point**: `HuggingFaceEndpoint` swaps local execution for a network call to a hosted endpoint
- **Key point**: The same `ChatHuggingFace` wrapper and LCEL chain code work unchanged

### 3. Groq
- **Key point**: `ChatGroq` is a native LangChain chat model — no extra wrapper needed
- **Key point**: All three backends plug into the identical `prompt | chat_model` LCEL pattern

### Next Steps
- `1.8_Package_Split_and_Imports_LangChain_v1.ipynb` — how these imports map onto LangChain 1.x's package split
